# CSV Basics — 02: Writing CSV Files

## What you will learn
Writing CSV seems trivial but has many important options that control
file size, compatibility, and downstream readability.

In this notebook:
1. Basic write with default options
2. Controlling output: single file vs multiple partitions
3. Custom separators, quotes, and null handling
4. Date and timestamp formatting
5. Header control and compatibility options
6. Writing to partitioned directories


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 08:59:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


In [2]:
# Create sample DataFrame to write
from pyspark.sql.types import *
import datetime

schema = StructType([
    StructField("order_id",    IntegerType()),
    StructField("customer",    StringType()),
    StructField("product",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("status",      StringType()),
    StructField("region",      StringType()),
    StructField("order_date",  DateType()),
])

data = [
    (1, "Alice",   "Laptop",      1, 999.99, "confirmed", "EMEA", datetime.date(2024,1,15)),
    (2, "Bob",     "Phone",       2, 599.99, "shipped",   "AMER", datetime.date(2024,1,16)),
    (3, "Carol",   "Tablet",      1, 349.99, "pending",   "APAC", datetime.date(2024,2,1)),
    (4, "Dave",    "Headphones",  3, 149.99, "delivered", "EMEA", datetime.date(2024,2,2)),
    (5, "Eve",     "Monitor",     2, 449.99, "confirmed", "AMER", datetime.date(2024,3,1)),
    (6, "Frank",   "Keyboard",    1,  89.99, "shipped",   "APAC", datetime.date(2024,3,15)),
    (7, "Grace",   "Mouse",       4,  49.99, "delivered", "EMEA", datetime.date(2024,3,20)),
    (8, "Henry",   "Webcam",      1,  79.99, "pending",   "AMER", datetime.date(2024,4,1)),
]
df = spark.createDataFrame(data, schema)
df.show()
print(f"DataFrame: {df.count()} rows, {len(df.columns)} columns")

+--------+--------+----------+--------+------+---------+------+----------+
|order_id|customer|   product|quantity| price|   status|region|order_date|
+--------+--------+----------+--------+------+---------+------+----------+
|       1|   Alice|    Laptop|       1|999.99|confirmed|  EMEA|2024-01-15|
|       2|     Bob|     Phone|       2|599.99|  shipped|  AMER|2024-01-16|
|       3|   Carol|    Tablet|       1|349.99|  pending|  APAC|2024-02-01|
|       4|    Dave|Headphones|       3|149.99|delivered|  EMEA|2024-02-02|
|       5|     Eve|   Monitor|       2|449.99|confirmed|  AMER|2024-03-01|
|       6|   Frank|  Keyboard|       1| 89.99|  shipped|  APAC|2024-03-15|
|       7|   Grace|     Mouse|       4| 49.99|delivered|  EMEA|2024-03-20|
|       8|   Henry|    Webcam|       1| 79.99|  pending|  AMER|2024-04-01|
+--------+--------+----------+--------+------+---------+------+----------+

DataFrame: 8 rows, 8 columns


## Step 1 — Default Write

By default Spark writes one file per partition (usually many small files).
The output directory contains:
- Part files: `part-00000-*.csv`  
- `_SUCCESS` marker file


In [3]:
import subprocess

OUT1 = f"{DATA_DIR}/write_default"

df.write.mode("overwrite").csv(OUT1)

# Check what was created
result = subprocess.run(["ls", "-la", OUT1], capture_output=True, text=True)
print("Default write output:")
print(result.stdout)

# Count part files
import glob
part_files = glob.glob(f"{OUT1}/part-*.csv")
print(f"Number of part files: {len(part_files)}")

# Read it back
df_back = spark.read.csv(OUT1, header=False, inferSchema=True)
print(f"\nRows written: {df_back.count()}")
df_back.show(3)
print("\nNote: no header! Default write does NOT include header row.")
print("Note: default separator is comma, no quotes around strings")

Default write output:
total 0
drwxr-xr-x 1 root root 4096 Apr  9 08:59 .
drwxr-xr-x 1 root root 4096 Apr  9 08:59 ..
-rw-r--r-- 1 root root   95 Apr  9 08:59 part-00000-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv
-rw-r--r-- 1 root root   12 Apr  9 08:59 .part-00000-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv.crc
-rw-r--r-- 1 root root  101 Apr  9 08:59 part-00001-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv
-rw-r--r-- 1 root root   12 Apr  9 08:59 .part-00001-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv.crc
-rw-r--r-- 1 root root   98 Apr  9 08:59 part-00002-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv
-rw-r--r-- 1 root root   12 Apr  9 08:59 .part-00002-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv.crc
-rw-r--r-- 1 root root   95 Apr  9 08:59 part-00003-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv
-rw-r--r-- 1 root root   12 Apr  9 08:59 .part-00003-a83698e8-857b-4e87-807e-3f4bf2b4ca26-c000.csv.crc
-rw-r--r-- 1 root root    0 Apr  9 08:59 _SUCCESS
-rw-r--r-- 1 root root    8 A


Rows written: 8
+---+-----+----------+---+------+---------+----+----------+
|_c0|  _c1|       _c2|_c3|   _c4|      _c5| _c6|       _c7|
+---+-----+----------+---+------+---------+----+----------+
|  3|Carol|    Tablet|  1|349.99|  pending|APAC|2024-02-01|
|  4| Dave|Headphones|  3|149.99|delivered|EMEA|2024-02-02|
|  5|  Eve|   Monitor|  2|449.99|confirmed|AMER|2024-03-01|
+---+-----+----------+---+------+---------+----+----------+
only showing top 3 rows

Note: no header! Default write does NOT include header row.
Note: default separator is comma, no quotes around strings


## Step 2 — Writing a Single File

For downstream compatibility (Excel, simple tools), you often need ONE file.
Use `coalesce(1)` before writing — merges all partitions to one.

⚠️ **Warning:** `coalesce(1)` sends all data to one task/executor.
Only use for small DataFrames (< 1 GB).


In [4]:
OUT2 = f"{DATA_DIR}/write_single"

# coalesce(1) = merge to 1 partition = 1 output file
df.coalesce(1).write.mode("overwrite").csv(OUT2, header=True)

part_files = glob.glob(f"{OUT2}/part-*.csv")
print(f"Single file write: {len(part_files)} part file(s)")

# Read back and verify
df_single = spark.read.csv(OUT2, header=True, inferSchema=True)
print(f"Rows: {df_single.count()}")
df_single.show()

# Show the actual file content
csv_file = part_files[0]
with open(csv_file) as f:
    print("\nActual file content:")
    print(f.read())

Single file write: 1 part file(s)
Rows: 8
+--------+--------+----------+--------+------+---------+------+----------+
|order_id|customer|   product|quantity| price|   status|region|order_date|
+--------+--------+----------+--------+------+---------+------+----------+
|       1|   Alice|    Laptop|       1|999.99|confirmed|  EMEA|2024-01-15|
|       2|     Bob|     Phone|       2|599.99|  shipped|  AMER|2024-01-16|
|       3|   Carol|    Tablet|       1|349.99|  pending|  APAC|2024-02-01|
|       4|    Dave|Headphones|       3|149.99|delivered|  EMEA|2024-02-02|
|       5|     Eve|   Monitor|       2|449.99|confirmed|  AMER|2024-03-01|
|       6|   Frank|  Keyboard|       1| 89.99|  shipped|  APAC|2024-03-15|
|       7|   Grace|     Mouse|       4| 49.99|delivered|  EMEA|2024-03-20|
|       8|   Henry|    Webcam|       1| 79.99|  pending|  AMER|2024-04-01|
+--------+--------+----------+--------+------+---------+------+----------+


Actual file content:
order_id,customer,product,quantity,

## Step 3 — Write Options: sep, quote, nullValue, dateFormat


In [5]:
OUT3 = f"{DATA_DIR}/write_options"

# DataFrame with null values
import datetime
df_with_nulls = spark.createDataFrame([
    (1, "Alice",  999.99, "confirmed", datetime.date(2024,1,15)),
    (2, None,     599.99, "shipped",   datetime.date(2024,2,1)),
    (3, "Carol",  None,   "pending",   None),
], ["order_id","customer","price","status","order_date"])

df_with_nulls.coalesce(1).write.mode("overwrite").csv(
    OUT3,
    header        = True,
    sep           = ";",            # semicolon separator (European format)
    quote         = '"',            # quote strings with double-quote
    quoteAll      = True,           # quote ALL string values (not just when needed)
    nullValue     = "NULL",         # write null as "NULL" string
    dateFormat    = "dd/MM/yyyy",   # European date format
    emptyValue    = "",             # empty string for empty values
    compression   = "none",        # no compression for human-readable
)

csv_file = glob.glob(f"{OUT3}/part-*.csv")[0]
print("CSV with custom options:")
with open(csv_file) as f:
    print(f.read())

CSV with custom options:
"order_id";"customer";"price";"status";"order_date"
"1";"Alice";"999.99";"confirmed";"15/01/2024"
"2";"NULL";"599.99";"shipped";"01/02/2024"
"3";"Carol";"NULL";"pending";"NULL"



## Step 4 — Timestamp Formatting


In [6]:
import datetime

df_ts = spark.createDataFrame([
    (1, datetime.datetime(2024, 1, 15, 10, 30, 0),   "login"),
    (2, datetime.datetime(2024, 1, 15, 10, 35, 22),  "purchase"),
    (3, datetime.datetime(2024, 1, 15, 11, 0,  0),   "logout"),
], ["event_id","event_ts","event_type"])

OUT4 = f"{DATA_DIR}/write_timestamps"

# Default timestamp format
df_ts.coalesce(1).write.mode("overwrite").csv(f"{OUT4}/default", header=True)

# Custom timestamp format
df_ts.coalesce(1).write.mode("overwrite").csv(
    f"{OUT4}/custom", header=True,
    timestampFormat = "dd/MM/yyyy HH:mm:ss"
)

for fmt in ["default", "custom"]:
    csv_file = glob.glob(f"{OUT4}/{fmt}/part-*.csv")[0]
    print(f"\n{fmt} format:")
    with open(csv_file) as f:
        print(f.read())


default format:
event_id,event_ts,event_type
1,2024-01-15T10:30:00.000Z,login
2,2024-01-15T10:35:22.000Z,purchase
3,2024-01-15T11:00:00.000Z,logout


custom format:
event_id,event_ts,event_type
1,15/01/2024 10:30:00,login
2,15/01/2024 10:35:22,purchase
3,15/01/2024 11:00:00,logout



## Step 5 — Save Modes

Spark has 4 save modes that control what happens if the output path already exists:

| Mode | Behavior |
|---|---|
| `overwrite` | Delete existing data and write new |
| `append` | Add new files to existing directory |
| `ignore` | Do nothing if path exists (silently skip) |
| `error` (default) | Raise error if path exists |


In [7]:
OUT5 = f"{DATA_DIR}/write_modes"

print("Mode: overwrite")
df.coalesce(1).write.mode("overwrite").csv(OUT5, header=True)
count1 = spark.read.csv(OUT5, header=True).count()
print(f"  After first write: {count1} rows")

df.coalesce(1).write.mode("overwrite").csv(OUT5, header=True)
count2 = spark.read.csv(OUT5, header=True).count()
print(f"  After overwrite  : {count2} rows (same data, old deleted)")

print("\nMode: append")
df.coalesce(1).write.mode("append").csv(OUT5, header=True)
count3 = spark.read.csv(OUT5, header=True).count()
print(f"  After append     : {count3} rows (doubled!)")

print("\nMode: ignore")
df.coalesce(1).write.mode("ignore").csv(OUT5, header=True)
count4 = spark.read.csv(OUT5, header=True).count()
print(f"  After ignore     : {count4} rows (no change)")

print("\nMode: error (default)")
try:
    df.coalesce(1).write.csv(OUT5, header=True)
except Exception as e:
    print(f"  Error raised: {type(e).__name__} — path already exists")

Mode: overwrite
  After first write: 8 rows


  After overwrite  : 8 rows (same data, old deleted)

Mode: append
  After append     : 16 rows (doubled!)

Mode: ignore
  After ignore     : 16 rows (no change)

Mode: error (default)
  Error raised: AnalysisException — path already exists


## Step 6 — Writing Partitioned CSV

For large exports partitioned by a column, use `partitionBy`.
This creates a directory structure like `region=EMEA/part-00000.csv`.


In [8]:
OUT6 = f"{DATA_DIR}/write_partitioned"

df.write.mode("overwrite") \
        .partitionBy("region") \
        .csv(OUT6, header=True)

# Show directory structure
import os
for root, dirs, files in os.walk(OUT6):
    level = root.replace(OUT6, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

print("\nRead specific partition (EMEA only — partition pruning):")
df_emea = spark.read.csv(f"{OUT6}/region=EMEA", header=True, inferSchema=True)
df_emea.show()

write_partitioned/
  ._SUCCESS.crc
  _SUCCESS
  region=AMER/
    .part-00000-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    .part-00002-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    .part-00003-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    part-00000-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
    part-00002-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
    part-00003-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
  region=APAC/
    .part-00001-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    .part-00002-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    part-00001-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
    part-00002-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
  region=EMEA/
    .part-00000-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    .part-00001-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    .part-00003-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv.crc
    part-00000-e65ed849-7c0b-43d8-b94b-5ca443562232.c000.csv
    part-00001-

## Summary: CSV Write Options Cheat Sheet

```python
df.write
  .mode("overwrite")         # overwrite | append | ignore | error
  .csv(
    path,
    header          = True,
    sep             = ",",
    quote           = '"',
    quoteAll        = False,   # True = quote all strings
    escape          = "\\",
    nullValue       = "",      # how to represent null
    emptyValue      = "",
    dateFormat      = "yyyy-MM-dd",
    timestampFormat = "yyyy-MM-dd'T'HH:mm:ss",
    compression     = "none",  # none|gzip|bz2|deflate
    lineSep         = "\n",
  )
```

**Common patterns:**
- Single file for tools: `df.coalesce(1).write.csv(path, header=True)`
- Partitioned export: `df.write.partitionBy("date").csv(path, header=True)`
- Compressed: `.csv(path, compression="gzip")` — creates `.csv.gz` files
